## Import Libraries and Datasets

In [121]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
sorted(p.name for p in DATA_DIR.iterdir())

['bars_1m.parquet',
 'events.json',
 'exchange_calendar.csv',
 'regime_labels.csv',
 'sim_runs.json',
 'stream_bars.json',
 'symbol_history.csv',
 'symbols.csv',
 'vendor_daily.csv']

In [122]:
exchange_calendar = pd.read_csv(DATA_DIR / "exchange_calendar.csv")
symbol_history = pd.read_csv(DATA_DIR / "symbol_history.csv")
symbols = pd.read_csv(DATA_DIR / "symbols.csv")
vendor_daily = pd.read_csv(DATA_DIR / "vendor_daily.csv")
bars = pd.read_parquet(DATA_DIR / "bars_1m.parquet")

exchange_calendar["_source_row"] = range(len(exchange_calendar))
symbol_history["_source_row"] = range(len(symbol_history))
symbols["_source_row"] = range(len(symbols))
vendor_daily["_source_row"] = range(len(vendor_daily))
bars["_source_row"] = range(len(bars))

for name, df in [
    ("exchange_calendar", exchange_calendar),
    ("symbol_history", symbol_history),
    ("symbols", symbols),
    ("vendor_daily", vendor_daily),
    ("bars", bars),
]:
    print(f"{name:16s} {df.shape}  {list(df.columns)[:6]} ...")

exchange_calendar (129, 7)  ['date', 'is_trading_day', 'session_open', 'session_close', 'timezone', 'note'] ...
symbol_history   (3, 6)  ['security_id', 'change_type', 'old_value', 'new_value', 'effective_date', '_source_row'] ...
symbols          (31, 8)  ['security_id', 'ticker', 'company_name', 'sector', 'listing_date', 'delisting_date'] ...
vendor_daily     (10831, 9)  ['run_id', 'date', 'ticker', 'open', 'high', 'low'] ...
bars             (4276175, 10)  ['run_id', 'ticker', 'ts', 'open', 'high', 'low'] ...


## Flatten nested JSON

In [123]:
import json

with open(DATA_DIR / "sim_runs.json", encoding="utf-8") as f:
    sim_runs_raw = json.load(f)

with open(DATA_DIR / "events.json", encoding="utf-8") as f:
    events_raw = json.load(f)

In [124]:
sim_runs_flat = pd.json_normalize(sim_runs_raw, sep="_")
sim_runs_flat.insert(0, "source_row", range(1, len(sim_runs_flat) + 1))

events_flat = pd.json_normalize(events_raw["events"], sep="_")
events_flat.insert(0, "source_row", range(1, len(events_flat) + 1))

In [125]:
print("SIM_RUNS")
print(sim_runs_flat.shape)
print(sim_runs_flat.columns.tolist())
display(sim_runs_flat.head())

print("\nEVENTS")
print(events_flat.shape)
print(events_flat.columns.tolist())
display(events_flat.head())

SIM_RUNS
(4, 25)
['source_row', 'run_id', 'config_version', 'created_at', 'seed', 'window_start', 'window_end', 'universe_stocks', 'universe_index_ticker', 'universe_index_method', 'universe_index_base_level', 'parameters_volatility_sigma_annual', 'parameters_climate_states', 'parameters_climate_variance_multiplier', 'parameters_climate_transition_matrix', 'parameters_news_memory_alpha', 'parameters_news_memory_beta', 'parameters_news_memory_gamma', 'parameters_intraday_open_boost', 'parameters_intraday_close_boost', 'parameters_shocks_distribution', 'parameters_shocks_df', 'parameters_jumps_rate_per_day', 'parameters_factors_market_weight', 'parameters_factors_sector_share']


,source_row,run_id,config_version,created_at,seed,window_start,window_end,universe_stocks,universe_index_ticker,universe_index_method,...,parameters_news_memory_alpha,parameters_news_memory_beta,parameters_news_memory_gamma,parameters_intraday_open_boost,parameters_intraday_close_boost,parameters_shocks_distribution,parameters_shocks_df,parameters_jumps_rate_per_day,parameters_factors_market_weight,parameters_factors_sector_share
0,1,RUN-2025A,1.4.0,2025-07-02T08:00:00+02:00,8522640,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
1,2,RUN-2025B,1.4.1,2025-07-02T08:00:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
2,3,RUN-2025C,1.4.0,2025-07-02T08:00:00+02:00,8325121,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
3,4,RUN-2025B,1.4.1,2025-07-03T11:20:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25



EVENTS
(62, 8)
['source_row', 'event_id', 'type', 'time', 'security_ticker', 'details_session', 'details_duration_minutes', 'details_ratio']


,source_row,event_id,type,time,security_ticker,details_session,details_duration_minutes,details_ratio
0,1,EV0001,earnings,2025-01-10T09:30:00-05:00,DALN,pre_open,NaN,NaN
1,2,EV0002,earnings,2025-01-14T09:30:00-05:00,BRIS,pre_open,NaN,NaN
2,3,EV0003,earnings,2025-01-15T09:30:00-05:00,SOLK,pre_open,NaN,NaN
3,4,EV0004,earnings,2025-01-24T09:30:00-05:00,XERN,pre_open,NaN,NaN
4,5,EV0005,earnings,2025-01-30T09:30:00-05:00,VALX,pre_open,NaN,NaN


In [126]:
bars.head()

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0
1,RUN-2025A,ARVT,2025-01-02 09:31:00,119.93,119.98,119.35,119.39,45001,2025-01-02 14:32:01+00:00,1
2,RUN-2025A,ARVT,2025-01-02 09:32:00,119.39,119.66,119.30,119.53,5825,2025-01-02 14:33:00+00:00,2
3,RUN-2025A,ARVT,2025-01-02 09:33:00,119.53,119.57,119.23,119.35,8195,2025-01-02 14:34:00+00:00,3
4,RUN-2025A,ARVT,2025-01-02 09:34:00,119.35,119.44,119.33,119.38,5631,2025-01-02 14:35:01+00:00,4


In [127]:
sim_runs_flat.head()

,source_row,run_id,config_version,created_at,seed,window_start,window_end,universe_stocks,universe_index_ticker,universe_index_method,...,parameters_news_memory_alpha,parameters_news_memory_beta,parameters_news_memory_gamma,parameters_intraday_open_boost,parameters_intraday_close_boost,parameters_shocks_distribution,parameters_shocks_df,parameters_jumps_rate_per_day,parameters_factors_market_weight,parameters_factors_sector_share
0,1,RUN-2025A,1.4.0,2025-07-02T08:00:00+02:00,8522640,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
1,2,RUN-2025B,1.4.1,2025-07-02T08:00:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
2,3,RUN-2025C,1.4.0,2025-07-02T08:00:00+02:00,8325121,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.03,0.8,0.6,1.2,0.6,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25
3,4,RUN-2025B,1.4.1,2025-07-03T11:20:00+02:00,7737823,2025-01-02,2025-06-30,30,INDEX,cap_weighted,...,0.33,0.8,0.0,0.0,0.0,student_t,4,0.02,"[0.12, 0.22, 0.45]",0.25


### Functions

In [128]:
# Rejection 
def make_rejects(df, mask, reason, source_file):
    bad_rows = df.loc[mask]

    reject_rows = [
        {
            "raw_payload": row.drop(labels="_source_row").to_dict(),
            "source_file": source_file,
            "source_row": int(row["_source_row"]),
            "reason": reason,
        }
        for _, row in bad_rows.iterrows()
    ]

    return pd.DataFrame(reject_rows)


all_rejects = []


In [129]:
exchange_calendar.head()

,date,is_trading_day,session_open,session_close,timezone,note,_source_row
0,2025-01-02,True,09:30,16:00,America/New_York,NaN,0
1,2025-01-03,True,09:30,16:00,America/New_York,NaN,1
2,2025-01-06,True,09:30,16:00,America/New_York,NaN,2
3,2025-01-07,True,09:30,16:00,America/New_York,NaN,3
4,2025-01-08,True,09:30,16:00,America/New_York,NaN,4


### EDA bar_1m

In [130]:
bars_cleaned = bars.copy()

In [131]:
bars.describe().style.format("{:,.2f}")

,open,high,low,close,volume,_source_row
count,"4,276,175.00","4,276,175.00","4,276,175.00","4,275,321.00","4,276,175.00","4,276,175.00"
mean,149.36,149.51,149.21,149.36,"8,358.85","2,138,087.00"
std,183.48,183.60,183.36,183.48,"25,639.99","1,234,425.54"
min,4.81,4.84,4.80,4.81,"-47,568.00",0.00
25%,63.54,63.59,63.48,63.54,"1,599.00","1,069,043.50"
50%,113.11,113.25,112.97,113.11,"2,825.00","2,138,087.00"
75%,157.54,157.75,157.35,157.54,"5,232.00","3,207,130.50"
max,"1,483.31","1,483.84","1,482.76","1,483.31","13,208,790.00","4,276,174.00"


#### Missing Values

In [132]:
bars.isna().sum()

run_id           0
ticker           0
ts               0
open             0
high             0
low              0
close          854
volume           0
received_at      0
_source_row      0
dtype: int64

In [133]:
mask = bars["close"].isna()
all_rejects.append(make_rejects(bars, mask, "MISSING_REQUIRED_FIELD", "bars_1m.parquet"))
bars_cleaned = bars.loc[~mask]

##### Negative Values

In [134]:
#-ve values
negative_value=(bars[["open","high","low","close","volume"]] <= 0).any(axis=1) #  at least one value in this row True
negative_value.sum()

np.int64(853)

In [135]:
mask = negative_value
all_rejects.append(make_rejects(bars, mask, "INVALID_NUMERIC_RANGE", "bars_1m.parquet"))
bars_cleaned = bars.loc[~mask]

#### Invalid OHLC

In [136]:
# OHLC consistency (high must be highest, low must be lowest)
bad_ohlc = (
    (bars.high < bars.low) |
    (bars.high < bars[["open","close"]].max(axis=1)) |
    (bars.low  > bars[["open","close"]].min(axis=1))
)
bad_ohlc.sum()

np.int64(1282)

In [137]:
# Invalid OHLC
mask = bad_ohlc
all_rejects.append(make_rejects(bars, mask, "INVALID_OHLC", "bars_1m.parquet"))
bars_cleaned = bars.loc[~mask]


#### Duplicates

In [138]:
# duplicated records
bars.duplicated(subset=["run_id","ticker","ts"]).sum()

np.int64(15120)

In [139]:
# Duplicates (keep the earliest received_at)
bars = bars.sort_values("received_at")
mask = bars.duplicated(["run_id", "ticker", "ts"], keep="first")
all_rejects.append(make_rejects(bars, mask, "DUPLICATE_BAR", "bars_1m.parquet"))
bars_cleaned = bars.loc[~mask]



#### Unknown Rows (Run Test)

In [140]:
bars["run_id"].value_counts()

run_id
RUN-2025A    1431358
RUN-2025B    1424991
RUN-2025C    1419566
RUN-TEST         260
Name: count, dtype: int64

In [141]:
mask = bars["run_id"] == "RUN-TEST"
all_rejects.append(make_rejects(bars, mask, "UNKNOWN_RUN", "bars_1m.parquet"))
bars_cleaned = bars.loc[~mask]

#### Fake Nulls

In [142]:
FAKE_NULLS = {"", "null", "none", "nan", "na", "n/a", "-", "?"}

for col in ["run_id", "ticker", "ts"]:
    s = bars[col].astype("string").str.strip().str.lower()
    print(col, s.isin(FAKE_NULLS).sum())

run_id 0
ticker 0
ts 0


#### Timestamp Check

In [143]:
bars["ts"].str.len().value_counts()

ts
19    2851052
20    1424849
16        159
10        115
Name: count, dtype: int64

In [162]:
bars.assign(ts_len=bars["ts"].str.len()).groupby("ts_len", dropna=False).head(3).sort_values("ts_len")

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_local,ts_flag,ts_len
578109,RUN-2025A,MARB,not-a-time,145.35,145.42,145.23,145.26,3640,2025-01-02 15:46:01+00:00,6572,NaT,None,10
1377917,RUN-2025A,INDEX,not-a-time,988.70,988.75,988.34,988.54,129105,2025-01-02 16:31:02+00:00,10550,NaT,None,10
1911069,RUN-2025B,KELT,not-a-time,88.05,88.10,87.98,88.01,3728,2025-01-02 17:07:01+00:00,13644,NaT,None,10
1767283,RUN-2025B,HALS,31/03/2025 10:15,58.70,58.73,58.69,58.71,4639,2025-01-02 18:47:01+00:00,22393,2025-03-31 10:15:00,TS_REFORMATTED_DMY,16
2534653,RUN-2025B,XERN,31/03/2025 10:15,17.78,17.78,17.77,17.77,3824,2025-01-02 19:03:02+00:00,23801,2025-03-31 10:15:00,TS_REFORMATTED_DMY,16
2662373,RUN-2025B,ASTR,31/03/2025 10:15,162.12,162.14,162.02,162.02,5536,2025-01-03 16:12:02+00:00,42924,2025-03-31 10:15:00,TS_REFORMATTED_DMY,16
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,1,2025-01-02 09:30:00,None,19
578034,RUN-2025A,MARB,2025-01-02 09:30:00,146.30,146.42,146.29,146.38,4849,2025-01-02 14:31:00+00:00,2,2025-01-02 09:30:00,None,19
3047740,RUN-2025C,ERTL,2025-01-02 09:30:00,117.22,117.81,117.10,117.46,7116,2025-01-02 14:31:00+00:00,7,2025-01-02 09:30:00,None,19
2246627,RUN-2025B,ROSS,2025-01-02T14:30:00Z,35.87,35.88,35.85,35.87,896,2025-01-02 14:31:00+00:00,3,2025-01-02 09:30:00,TS_CONVERTED_FROM_UTC,20


In [145]:
exchange_calendar.timezone.unique()

<ArrowStringArray>
['America/New_York']
Length: 1, dtype: str

In [146]:
TZ = "America/New_York"    # based on exchange_calendar.csv (timezone column)

ts = bars["ts"].astype("string").str.strip()

# Format A: 2025-01-02 09:30:00  (already local)
fmt_local = pd.to_datetime(ts, format="%Y-%m-%d %H:%M:%S", errors="coerce")

# Format B: 2025-01-02T14:30:00Z  (UTC → convert to local)
fmt_utc = pd.to_datetime(ts, format="%Y-%m-%dT%H:%M:%SZ", errors="coerce", utc=True)
fmt_utc = fmt_utc.dt.tz_convert(TZ).dt.tz_localize(None)

# Format C: 31/03/2025 10:15  (day/month/year) 
fmt_dmy = pd.to_datetime(ts, format="%d/%m/%Y %H:%M", errors="coerce")

# Combine: take the first format that worked
bars_cleaned["ts_local"] = fmt_local.fillna(fmt_utc).fillna(fmt_dmy)

# Remember what was converted (quality flag)
bars["ts_local"] = fmt_local.fillna(fmt_utc).fillna(fmt_dmy)
bars["ts_flag"] = None
bars.loc[fmt_utc.notna(), "ts_flag"] = "TS_CONVERTED_FROM_UTC"
bars.loc[fmt_dmy.notna(), "ts_flag"] = "TS_REFORMATTED_DMY"

In [147]:
# Fixed dates
display(bars_cleaned.loc[bars_cleaned["ts_local"].isna(), "ts"].value_counts())

ts
2025-02-30 10:00:00    151
not-a-time             115
Name: count, dtype: int64

In [148]:
mask = bars["ts_local"].isna()
all_rejects.append(make_rejects(bars, mask, "MALFORMED_TIMESTAMP", "bars_1m.parquet"))
bars_cleaned = bars.loc[~mask]

In [149]:
# 2. compare with received_at: lateness should be small and positive for every format

bars_cleaned["end_utc"] = (bars_cleaned["ts_local"] + pd.Timedelta(minutes=1)) \
            .dt.tz_localize(TZ, ambiguous="NaT", nonexistent="NaT") \
            .dt.tz_convert("UTC")                        # 2. safe around clock changes

bars_cleaned["lateness_s"] = (bars_cleaned["received_at"] - end_utc).dt.total_seconds()

ts_len = bars_cleaned["ts"].astype("string").str.strip().str.len()   # 3. computed here, so it always exists
display(bars_cleaned["lateness_s"].groupby(ts_len).median())

ts
16    88322.0
19        1.0
20        1.0
Name: lateness_s, dtype: float64

In [150]:
# Placeholder 
g = bars_cleaned[ts_len == 16]
g["ts"].value_counts().head(10)


ts
31/03/2025 10:15    159
Name: count, dtype: int64

In [151]:
g

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_local,ts_flag,end_utc,lateness_s
1767283,RUN-2025B,HALS,31/03/2025 10:15,58.70,58.73,58.69,58.71,4639,2025-01-02 18:47:01+00:00,1767283,2025-03-31 10:15:00,TS_REFORMATTED_DMY,2025-03-31 14:16:00+00:00,-7586939.0
2534653,RUN-2025B,XERN,31/03/2025 10:15,17.78,17.78,17.77,17.77,3824,2025-01-02 19:03:02+00:00,2534653,2025-03-31 10:15:00,TS_REFORMATTED_DMY,2025-03-31 14:16:00+00:00,-7585978.0
2662373,RUN-2025B,ASTR,31/03/2025 10:15,162.12,162.14,162.02,162.02,5536,2025-01-03 16:12:02+00:00,2662373,2025-03-31 10:15:00,TS_REFORMATTED_DMY,2025-03-31 14:16:00+00:00,-7509838.0
337686,RUN-2025A,HALS,31/03/2025 10:15,60.15,60.24,60.15,60.20,4523,2025-01-03 16:42:01+00:00,337686,2025-03-31 10:15:00,TS_REFORMATTED_DMY,2025-03-31 14:16:00+00:00,-7508039.0
1863814,RUN-2025B,JORV,31/03/2025 10:15,461.19,461.24,461.11,461.19,417,2025-01-06 15:53:00+00:00,1863814,2025-03-31 10:15:00,TS_REFORMATTED_DMY,2025-03-31 14:16:00+00:00,-7251780.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1622188,RUN-2025B,DALN,31/03/2025 10:15,127.82,128.01,127.75,127.94,11044,2025-06-26 15:55:00+00:00,1622188,2025-03-31 10:15:00,TS_REFORMATTED_DMY,2025-03-31 14:16:00+00:00,7522740.0
2389708,RUN-2025B,TAVV,31/03/2025 10:15,110.61,110.89,110.54,110.68,957,2025-06-26 19:46:02+00:00,2389708,2025-03-31 10:15:00,TS_REFORMATTED_DMY,2025-03-31 14:16:00+00:00,7536602.0
3142724,RUN-2025C,FENM,31/03/2025 10:15,39.24,39.26,39.24,39.26,1092,2025-06-27 17:01:02+00:00,3142724,2025-03-31 10:15:00,TS_REFORMATTED_DMY,2025-03-31 14:16:00+00:00,7613102.0
336960,RUN-2025A,GALR,31/03/2025 10:15,13.81,13.86,13.81,13.86,17216,2025-06-30 16:36:01+00:00,336960,2025-03-31 10:15:00,TS_REFORMATTED_DMY,2025-03-31 14:16:00+00:00,7870801.0


In [152]:
placeholder = bars_cleaned["ts"].astype("string").str.strip() == "31/03/2025 10:15"
mask = placeholder
all_rejects.append(make_rejects(bars_cleaned, mask, "MALFORMED_TIMESTAMP", "bars_1m.parquet"))
bars_cleaned = bars_cleaned.loc[~mask]
print(mask.sum())    # 159

159


In [153]:
## Adding columns
bars_cleaned["trading_date"] = bars_cleaned["ts_local"].dt.date
bars_cleaned["bar_time"]     = bars_cleaned["ts_local"].dt.time

In [154]:
bars_cleaned

,run_id,ticker,ts,open,high,low,close,volume,received_at,_source_row,ts_local,ts_flag,end_utc,lateness_s,trading_date,bar_time
0,RUN-2025A,ARVT,2025-01-02 09:30:00,120.07,120.08,119.84,119.93,8570,2025-01-02 14:31:00+00:00,0,2025-01-02 09:30:00,None,2025-01-02 14:31:00+00:00,0.0,2025-01-02,09:30:00
578034,RUN-2025A,MARB,2025-01-02 09:30:00,146.30,146.42,146.29,146.38,4849,2025-01-02 14:31:00+00:00,578034,2025-01-02 09:30:00,None,2025-01-02 14:31:00+00:00,0.0,2025-01-02,09:30:00
2246627,RUN-2025B,ROSS,2025-01-02T14:30:00Z,35.87,35.88,35.85,35.87,896,2025-01-02 14:31:00+00:00,2246627,2025-01-02 09:30:00,TS_CONVERTED_FROM_UTC,2025-01-02 14:31:00+00:00,0.0,2025-01-02,09:30:00
2342547,RUN-2025B,TAVV,2025-01-02T14:30:00Z,143.02,143.07,142.96,143.05,1377,2025-01-02 14:31:00+00:00,2342547,2025-01-02 09:30:00,TS_CONVERTED_FROM_UTC,2025-01-02 14:31:00+00:00,0.0,2025-01-02,09:30:00
2438467,RUN-2025B,VALX,2025-01-02T14:30:00Z,227.66,227.90,227.60,227.79,4109,2025-01-02 14:31:00+00:00,2438467,2025-01-02 09:30:00,TS_CONVERTED_FROM_UTC,2025-01-02 14:31:00+00:00,0.0,2025-01-02,09:30:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2952181,RUN-2025C,BELX,2025-06-30 15:57:00,183.68,183.91,183.55,183.74,2038,2025-06-30 20:11:23+00:00,2952181,2025-06-30 15:57:00,None,2025-06-30 19:58:00+00:00,803.0,2025-06-30,15:57:00
3668801,RUN-2025C,ORBX,2025-06-30 15:55:00,158.59,158.98,158.47,158.96,3746,2025-06-30 20:11:46+00:00,3668801,2025-06-30 15:55:00,None,2025-06-30 19:56:00+00:00,946.0,2025-06-30,15:55:00
2999952,RUN-2025C,CORB,2025-06-30 15:47:00,87.56,87.83,87.47,87.72,6023,2025-06-30 20:12:43+00:00,2999952,2025-06-30 15:47:00,None,2025-06-30 19:48:00+00:00,1483.0,2025-06-30,15:47:00
1011560,RUN-2025A,UMBT,2025-06-30 15:56:00,103.33,103.33,103.22,103.23,771,2025-06-30 20:16:55+00:00,1011560,2025-06-30 15:56:00,None,2025-06-30 19:57:00+00:00,1195.0,2025-06-30,15:56:00


#### Cleansing Results

In [155]:
# Results
rejects = pd.concat(all_rejects, ignore_index=True)
clean = df

print(rejects["reason"].value_counts())

reason
DUPLICATE_BAR             15120
INVALID_OHLC               1282
MISSING_REQUIRED_FIELD      854
INVALID_NUMERIC_RANGE       853
MALFORMED_TIMESTAMP         425
UNKNOWN_RUN                 260
Name: count, dtype: int64


### Rejection Function

RUN-2025B appears twice in sim_runs; same config_version, but created_at differs. 
For the bar join, should retain one run record and exclude conflicting "created_at" as the relation is many_to_one

In [156]:
sim_runs_clean = (
    sim_runs_flat[["run_id", "config_version"]]
    .drop_duplicates(subset=["run_id", "config_version"])
)

In [157]:
bars_with_runs= bars.merge(
    sim_runs_clean,
    on="run_id",
    how="left",
    validate="many_to_one"
)

In [158]:
print("bars before:", len(bars))
print("bars after: ", len(bars_with_runs))
print("unmatched bars:", bars_with_runs["config_version"].isna().sum())

bars before: 4276175
bars after:  4276175
unmatched bars: 260


In [159]:
bars_with_runs.loc[
    bars_with_runs["config_version"].isna(),
    "run_id"
].drop_duplicates()

102168    RUN-TEST
Name: run_id, dtype: str

### Bars with an unknown run

**Finding:** The bars table contains 260 rows with `run_id = RUN-TEST`. This ID does not appear in the flattened `sim_runs` table.

**How it was detected:** I compared the distinct `run_id` values in `bars` with those in `sim_runs_flat`. The `RUN-TEST` bars did not match a known simulation run.

**Decision:** These bars are candidates for rejection because their run cannot be verified. For each rejected bar, the rejects table preserves the raw bar payload, source file (`bars_1m.parquet`), source row, and one rejection reason: `UNKNOWN_RUN`.

In [160]:
bars = bars.copy()
bars["_source_row"] = range(1, len(bars) + 1)

In [161]:
unknown_run_bars = bars.loc[bars["run_id"] == "RUN-TEST"]

reject_rows = [
    {
        "raw_payload": row.drop(labels="_source_row").to_dict(),
        "source_file": "bars_1m.parquet",
        "source_row": int(row["_source_row"]),
        "reason": "UNKNOWN_RUN",
    }
    for _, row in unknown_run_bars.iterrows()
]

rejects = pd.DataFrame(reject_rows)

display(rejects)

,raw_payload,source_file,source_row,reason
0,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,102169,UNKNOWN_RUN
1,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,102450,UNKNOWN_RUN
2,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,102695,UNKNOWN_RUN
3,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,102965,UNKNOWN_RUN
4,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,103201,UNKNOWN_RUN
...,...,...,...,...
255,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,169194,UNKNOWN_RUN
256,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,169471,UNKNOWN_RUN
257,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,169719,UNKNOWN_RUN
258,"{'run_id': 'RUN-TEST', 'ticker': 'BELX', 'ts':...",bars_1m.parquet,170006,UNKNOWN_RUN
